# Experiment No. 6 — Containerization & API Deployment

**Aim:** Containerization & API Deployment

**Objective:** Package the Experiment 4 model in Docker; build an API with FastAPI for predictions.

**Open-source tools:** Docker, FastAPI, Flask, Python

This notebook is self-contained — it loads `best_delivery_time_model.pkl` (the model
saved by Experiment 4) directly, so it does not depend on any other notebook's kernel
state.

> **Sandbox note:** `fastapi`/`uvicorn` cannot be installed offline here, and there is
> no Docker daemon in this sandbox. The prediction logic itself is executed for real
> below; the FastAPI wrapper and Dockerfile just expose that same, already-tested
> function over HTTP.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, json
import joblib
import pandas as pd

MODEL_PATH = "best_delivery_time_model.pkl"

if not os.path.exists(MODEL_PATH):
    try:
        from google.colab import files
        print("Please choose 'best_delivery_time_model.pkl' from your computer "
              "(the model saved by Experiment 4):")
        uploaded = files.upload()
        MODEL_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            "Place 'best_delivery_time_model.pkl' (from Experiment 4) in this notebook's "
            "working directory, or run this cell in Colab to upload it."
        )

model = joblib.load(MODEL_PATH)
print("Loaded model:", type(model.named_steps["model"]).__name__)

Loaded model: GradientBoostingRegressor


## 1. Write FastAPI `/predict` endpoint

**`api/model_utils.py`** — shared prediction logic (imported by both the API and the tests):

In [2]:
FEATURE_COLUMNS = [
    "Company", "City", "Customer_Age", "Age_Group", "Product_Category",
    "Items_Count", "Order_Size", "Order_Value", "Discount_Percent",
    "Payment_Method", "Distance_Km", "Delivery_Mode",
    "Customer_Rating", "Delivery_Partner_Rating",
]

def predict_from_json(payload: dict) -> dict:
    missing = [c for c in FEATURE_COLUMNS if c not in payload]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")
    row = pd.DataFrame([{c: payload[c] for c in FEATURE_COLUMNS}])
    pred = float(model.predict(row)[0])
    return {"predicted_delivery_time_min": round(pred, 2)}

print("predict_from_json() ready —", len(FEATURE_COLUMNS), "expected fields")

predict_from_json() ready — 14 expected fields


**`api/main.py`** — FastAPI app (not executed here — needs `pip install fastapi uvicorn`):

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from model_utils import predict_from_json

app = FastAPI(title="Delivery Time Predictor API", version="1.0.0")

class OrderRequest(BaseModel):
    Company: str
    City: str
    Customer_Age: int = Field(..., ge=15, le=100)
    Age_Group: str
    Product_Category: str
    Items_Count: int = Field(..., ge=1)
    Order_Size: str
    Order_Value: float = Field(..., ge=0)
    Discount_Percent: int = Field(..., ge=0, le=100)
    Payment_Method: str
    Distance_Km: float = Field(..., ge=0)
    Delivery_Mode: str
    Customer_Rating: float = Field(..., ge=1, le=6)
    Delivery_Partner_Rating: float = Field(..., ge=1, le=6)

class PredictionResponse(BaseModel):
    predicted_delivery_time_min: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionResponse)
def predict(order: OrderRequest):
    try:
        result = predict_from_json(order.dict())
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))
    return PredictionResponse(**result)

# Run with: uvicorn main:app --reload --port 8000

## 2. Test with sample JSON POST

In [3]:
sample_request = {
    "Company": "Blinkit", "City": "Bengaluru", "Customer_Age": 29, "Age_Group": "25-34",
    "Product_Category": "Groceries", "Items_Count": 8, "Order_Size": "Medium",
    "Order_Value": 950.0, "Discount_Percent": 10, "Payment_Method": "UPI",
    "Distance_Km": 4.2, "Delivery_Mode": "Bike", "Customer_Rating": 4.5,
    "Delivery_Partner_Rating": 4.2,
}

print("POST /predict request body:")
print(json.dumps(sample_request, indent=2))

response = predict_from_json(sample_request)
print("\nResponse:")
print(json.dumps(response, indent=2))

other = dict(sample_request, City="Delhi", Distance_Km=15.0, Order_Size="Bulk", Items_Count=18)
print("\nSecond sample ->", predict_from_json(other))

try:
    predict_from_json({"Company": "Zepto"})
except ValueError as e:
    print("\nValidation error correctly raised for an incomplete payload:")
    print(" ", e)

POST /predict request body:
{
  "Company": "Blinkit",
  "City": "Bengaluru",
  "Customer_Age": 29,
  "Age_Group": "25-34",
  "Product_Category": "Groceries",
  "Items_Count": 8,
  "Order_Size": "Medium",
  "Order_Value": 950.0,
  "Discount_Percent": 10,
  "Payment_Method": "UPI",
  "Distance_Km": 4.2,
  "Delivery_Mode": "Bike",
  "Customer_Rating": 4.5,
  "Delivery_Partner_Rating": 4.2
}

Response:
{
  "predicted_delivery_time_min": 22.69
}

Second sample -> {'predicted_delivery_time_min': 22.65}

Validation error correctly raised for an incomplete payload:
  Missing required fields: ['City', 'Customer_Age', 'Age_Group', 'Product_Category', 'Items_Count', 'Order_Size', 'Order_Value', 'Discount_Percent', 'Payment_Method', 'Distance_Km', 'Delivery_Mode', 'Customer_Rating', 'Delivery_Partner_Rating']


## 3. Write Dockerfile and build image

In [ ]:
"""
FROM python:3.11-slim
WORKDIR /app
COPY api/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt
COPY models/best_delivery_time_model.pkl /app/models/best_delivery_time_model.pkl
COPY api/main.py /app/main.py
COPY api/model_utils.py /app/model_utils.py
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
# Build:  docker build -f api/Dockerfile -t delivery-time-api .

## 4. Run and test local container

In [ ]:
# Run:   docker run -p 8000:8000 delivery-time-api
# Test:  curl -X POST http://localhost:8000/predict \
#          -H "Content-Type: application/json" -d @api/sample_request.json
#
# This sandbox has no Docker daemon and no internet access to pull the base
# image, so the container itself could not be built here. The exact code path
# it serves (predict_from_json, above) was already executed for real with a
# genuine sample request and response.

## Deliverables
- **API code** — `predict_from_json` (executed above) and the FastAPI wrapper.
- **Dockerfile** — above.
- **Local test evidence** — the real request/response pair and validation-error check in Step 2.

## Conclusion

The prediction logic the API exposes was executed directly, above, and returned a correct, well-formed prediction for a valid order (22.69 min), a similar prediction for a second order (consistent with Experiment 4's finding that the model's outputs cluster near the dataset mean), and correctly rejected an incomplete request. The FastAPI wrapper and Dockerfile add an HTTP interface and portable runtime around that same, already-tested function; building and running the container requires Docker, which isn't available in this sandbox, so that step should be completed in a Docker-enabled environment using the commands above.